In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- bench_interp_time_regular ---

# --- bench_interp_time_rmse ---

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_bench_interp_time_regular(df=None):
    if df is None:
        df = pd.DataFrame({"date":["2020-01-01"],"value":[1.0],"station_id":["s1"]})
    df = df.dropna()
    station_ids = df.station_id.tolist()
    first_station_id = station_ids[0]
    return df[df["station_id"] == first_station_id]
    return first_station_id

def before_bench_interp_time_rmse():
    def get_rmse(regular_values: pd.Series, interpolated_values: pd.Series):
        diff = (regular_values.reset_index(drop=True) - interpolated_values.reset_index(drop=True)).dropna()
        n = diff.size
        return ((diff**2).sum() / n) ** 0.5
    return get_rmse

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_bench_interp_time_regular(df=None):
    if df is None:
        df = pl.DataFrame({"date":pl.Series(["2020-01-01"],dtype=pl.Utf8),"value":[1.0],"station_id":["s1"]})
    pd = pl  # LLM used `import polars as pd`
    df = df.drop_nulls()
    station_ids = df["station_id"].to_list()
    first_station_id = station_ids[0]
    return df.filter(pd.col("station_id") == first_station_id)
    return first_station_id

def gen_bench_interp_time_rmse():
    def get_rmse(regular_values: pl.Series, interpolated_values: pl.Series):
        diff = (regular_values - interpolated_values).drop_nulls()
        n = diff.len()
        return ((diff.pow(2).sum() / n) ** 0.5)
    return get_rmse

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: bench_interp_time_rmse ===

# L1 smoke – generated
try:
    _r = gen_bench_interp_time_rmse()
    print("✅ L1 smoke gen_bench_interp_time_rmse: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_bench_interp_time_rmse: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_bench_interp_time_rmse()
    print("✅ L1 smoke before_bench_interp_time_rmse: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_bench_interp_time_rmse: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – execute returned RMSE helper.
try:
    _before_fn = before_bench_interp_time_rmse()
    _gen_fn = gen_bench_interp_time_rmse()
    _rb = _before_fn(pd.Series([1.0, 3.0, 5.0]), pd.Series([1.0, 1.0, 2.0]))
    _rg = _gen_fn(pl.Series([1.0, 3.0, 5.0]), pl.Series([1.0, 1.0, 2.0]))
    if np.isclose(_rb, _rg, equal_nan=True):
        print("✅ L2 equivalence bench_interp_time_rmse value: MATCH")
    else:
        print(f"❌ L2 equivalence bench_interp_time_rmse value: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence bench_interp_time_rmse: setup error — {type(_e).__name__}: {_e}")

# L3 edge – execute returned function with nulls and non-trivial values
try:
    _before_fn = before_bench_interp_time_rmse()
    _gen_fn = gen_bench_interp_time_rmse()
    _before_edge = _before_fn(
        pd.Series([1.0, np.nan, 3.0, 7.0]),
        pd.Series([1.0, 2.0, 5.0, 1.0]),
    )
    _gen_edge = _gen_fn(
        pl.Series([1.0, None, 3.0, 7.0]),
        pl.Series([1.0, 2.0, 5.0, 1.0]),
    )
    if np.isclose(_before_edge, _gen_edge, equal_nan=True):
        print("✅ L3 edge bench_interp_time_rmse: MATCH")
    else:
        print(f"❌ L3 edge bench_interp_time_rmse: MISMATCH — before={_before_edge}, gen={_gen_edge}")
except Exception as _e:
    print(f"❌ L3 edge bench_interp_time_rmse: {type(_e).__name__}: {_e}")
